In [0]:
# Feature Importance and Model Summary
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importance (coefficients)
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

# Display top features
top_features = feature_importance.head(10)
display(spark.createDataFrame(top_features))

# Confusion matrix for test set
cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

confusion_data = [
    ('True Negatives', int(tn)),
    ('False Positives', int(fp)),
    ('False Negatives', int(fn)),
    ('True Positives', int(tp))
]

confusion_df = spark.createDataFrame(confusion_data, ['Classification', 'Count'])
display(confusion_df)

# Model performance summary
performance_summary = f"""
Model Performance Summary:
- The model achieved an AUC of {test_auc:.4f} and Gini of {test_gini:.4f} on the test set
- High recall ({test_recall:.4f}) indicates good detection of default cases
- The WoE-encoded features (delinquency_category, interest_rate, income, age) significantly improved predictive power
- Feature engineering with binning and behavioral flags enhanced model interpretability
"""

feature,coefficient
delinquency_category_woe,-0.5580237617486781
income_bin_woe,-0.5271099211470006
num_hard_inquiries_6mo,0.2719213334190809
age_bin_woe,-0.2597244145832989
employment_length_years,-0.19883015608391819
num_delinquencies_2yr,0.19004776726454875
credit_utilisation_pct,0.18205368749158735
num_open_accounts,0.14477258804197388
high_utilization_flag,0.1428559489319835
dti_ratio,0.12100285374606659


Classification,Count
True Negatives,21974
False Positives,8549
False Negatives,1531
True Positives,4030


In [0]:
# Export loan_book_gold to CSV
df_export = spark.table('dataanalytics.ml1.loan_book_gold')

# Convert to Pandas
df_pandas = df_export.toPandas()

# Save to workspace
output_path = '/Workspace/Users/prosperproper700@gmail.com/loan_book_gold.csv'
df_pandas.to_csv(output_path, index=False)

export_summary = [
    ('File Location', output_path),
    ('Total Rows', len(df_pandas)),
    ('Total Columns', len(df_pandas.columns)),
    ('File Size (approx)', f'{len(df_pandas) * len(df_pandas.columns)} cells')
]

export_df = spark.createDataFrame(export_summary, ['Metric', 'Value'])
display(export_df)

Metric,Value
File Location,/Workspace/Users/prosperproper700@gmail.com/loan_book_gold.csv
Total Rows,120358
Total Columns,25
File Size (approx),3008950 cells


In [0]:
# Evaluate Model Performance
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, 
    f1_score, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt

# Make predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

y_pred_proba_train = model.predict_proba(X_train_scaled)[:, 1]
y_pred_proba_test = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics for training set
train_auc = roc_auc_score(y_train, y_pred_proba_train)
train_precision = precision_score(y_train, y_pred_train)
train_recall = recall_score(y_train, y_pred_train)
train_f1 = f1_score(y_train, y_pred_train)
train_gini = 2 * train_auc - 1

# Calculate metrics for test set
test_auc = roc_auc_score(y_test, y_pred_proba_test)
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_gini = 2 * test_auc - 1

# Log metrics to MLflow
with mlflow.start_run(run_id=run_id):
    # Training metrics
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('train_precision', train_precision)
    mlflow.log_metric('train_recall', train_recall)
    mlflow.log_metric('train_f1', train_f1)
    mlflow.log_metric('train_gini', train_gini)
    
    # Test metrics
    mlflow.log_metric('test_auc', test_auc)
    mlflow.log_metric('test_precision', test_precision)
    mlflow.log_metric('test_recall', test_recall)
    mlflow.log_metric('test_f1', test_f1)
    mlflow.log_metric('test_gini', test_gini)

# Create metrics summary
metrics_data = [
    ('AUC', f'{train_auc:.4f}', f'{test_auc:.4f}'),
    ('Precision', f'{train_precision:.4f}', f'{test_precision:.4f}'),
    ('Recall', f'{train_recall:.4f}', f'{test_recall:.4f}'),
    ('F1 Score', f'{train_f1:.4f}', f'{test_f1:.4f}'),
    ('Gini', f'{train_gini:.4f}', f'{test_gini:.4f}')
]

metrics_df = spark.createDataFrame(metrics_data, ['Metric', 'Train', 'Test'])
display(metrics_df)

Metric,Train,Test
AUC,0.7938,0.7950
Precision,0.3176,0.3204
Recall,0.7177,0.7247
F1 Score,0.4404,0.4443
Gini,0.5876,0.5900


In [0]:
# Train Logistic Regression Model
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

# Load gold dataset
df_model = spark.table('dataanalytics.ml1.loan_book_gold')

# Split into train and test
train_spark = df_model.filter(F.col('set') == 'train').drop('set', 'applicant_id_hash')
test_spark = df_model.filter(F.col('set') == 'test').drop('set', 'applicant_id_hash')

# Convert to pandas for sklearn
train_pd = train_spark.toPandas()
test_pd = test_spark.toPandas()

# Separate features and target
X_train = train_pd.drop('default_flag', axis=1)
y_train = train_pd['default_flag']
X_test = test_pd.drop('default_flag', axis=1)
y_test = test_pd['default_flag']

# Feature names
feature_names = X_train.columns.tolist()

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Start MLflow run
mlflow.set_experiment('/Users/prosperproper700@gmail.com/loan_default_model')

with mlflow.start_run(run_name='logistic_regression_woe_features') as run:
    # Log parameters
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('n_features', len(feature_names))
    mlflow.log_param('train_samples', len(X_train))
    mlflow.log_param('test_samples', len(X_test))
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('max_iter', 1000)
    
    # Train model
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    )
    
    model.fit(X_train_scaled, y_train)
    
    # Log model
    mlflow.sklearn.log_model(
        model, 
        'model',
        input_example=X_train_scaled[:5]
    )
    
    run_id = run.info.run_id

# Model trained successfully
model_info = [
    ('Run ID', run_id),
    ('Model Type', 'Logistic Regression'),
    ('Features', len(feature_names)),
    ('Training Samples', len(X_train)),
    ('Test Samples', len(X_test))
]

model_summary = spark.createDataFrame(model_info, ['Metric', 'Value'])
display(model_summary)

2026/05/14 16:57:52 INFO mlflow.tracking.fluent: Experiment with name '/Users/prosperproper700@gmail.com/loan_default_model' does not exist. Creating a new experiment.
2026/05/14 16:57:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-4225c63b-638a.cloud.databricks.com/ml/experiments/673025735347881/models/m-1fdfdcaccce94e7084b322fba602af87?o=7474645869796416


Metric,Value
Run ID,b89ad4562ce04ce7aa5ad0bb88c04d80
Model Type,Logistic Regression
Features,22
Training Samples,84274
Test Samples,36084


In [0]:
# Create final gold-layer dataset with selected features
# Select strong and medium features based on IV analysis

features_for_model = [
    # ID and target
    'applicant_id_hash', 'default_flag', 'set',
    
    # Strong numeric features
    'interest_rate', 'annual_income',
    
    # Medium numeric features  
    'age', 'num_delinquencies_2yr', 'months_since_oldest_account',
    'employment_length_years', 'months_since_last_delinquency',
    
    # Weak numeric features (still useful for model)
    'dti_ratio', 'credit_utilisation_pct', 'num_open_accounts',
    'loan_amount', 'num_hard_inquiries_6mo', 'total_revolving_balance',
    'pct_accounts_current',
    
    # WoE encoded categorical features (all strong)
    'delinquency_category_woe', 'age_bin_woe', 'income_bin_woe', 'interest_rate_bin_woe',
    
    # Behavioral flags
    'high_utilization_flag', 'new_account_flag', 'frequent_inquiries_flag', 'high_dti_flag'
]

# Create gold dataset
df_gold_final = df_gold.select(features_for_model)

# Save to gold table
df_gold_final.write.mode('overwrite').saveAsTable('dataanalytics.ml1.loan_book_gold')

# Summary
feature_count = len(features_for_model) - 3  # exclude ID, target, set
summary_data = [
    ('Total Features', feature_count),
    ('WoE Encoded Features', 4),
    ('Numeric Features', 14),
    ('Behavioral Flags', 4),
    ('Total Records', df_gold_final.count())
]

summary_result = spark.createDataFrame(summary_data, ['Metric', 'Value'])
display(summary_result)

Metric,Value
Total Features,22
WoE Encoded Features,4
Numeric Features,14
Behavioral Flags,4
Total Records,120358


In [0]:
# Apply WoE encoding to categorical features using joins
from pyspark.sql import DataFrame

def apply_woe_encoding_with_joins(df, woe_dict):
    """
    Apply WoE encoding to categorical features using joins (serverless compatible)
    """
    result_df = df
    
    for feature, woe_df in woe_dict.items():
        # Select only the feature and woe columns
        woe_lookup = woe_df.select(
            F.col(feature).alias(f'{feature}_lookup'),
            F.col('woe').alias(f'{feature}_woe')
        )
        
        # Join to apply WoE encoding
        result_df = result_df.join(
            woe_lookup,
            result_df[feature] == woe_lookup[f'{feature}_lookup'],
            'left'
        ).drop(f'{feature}_lookup')
        
        # Fill nulls with 0 (for categories not in training set)
        result_df = result_df.withColumn(f'{feature}_woe', 
            F.coalesce(F.col(f'{feature}_woe'), F.lit(0.0)))
    
    return result_df

# Apply WoE encoding
df_gold = apply_woe_encoding_with_joins(df_features, woe_results)

# Display WoE values for delinquency_category
display(woe_results['delinquency_category'].select('delinquency_category', 'woe', 'iv_contrib').orderBy(F.desc('woe')))

delinquency_category,woe,iv_contrib
No_History,0.737265873060947,0.20895858611605864
Old_36mo+,-0.02121930202632277,5.4151149302932766E-5
Moderate_12-36mo,-0.3472241949297031,0.01819431653659985
Recent_0-12mo,-0.7317818436858982,0.16719191160177194


In [0]:
# Calculate WoE (Weight of Evidence) and IV (Information Value) for categorical features
import math

def calculate_woe_iv(df, feature_col, target_col='default_flag'):
    """
    Calculate WoE and IV for a categorical feature
    """
    # Filter to training set only for WoE calculation
    train_df = df.filter(F.col('set') == 'train')
    
    # Calculate totals
    total_goods = train_df.filter(F.col(target_col) == 0).count()
    total_bads = train_df.filter(F.col(target_col) == 1).count()
    
    # Group by feature and calculate counts
    woe_df = train_df.groupBy(feature_col).agg(
        F.count('*').alias('total'),
        F.sum(F.when(F.col(target_col) == 0, 1).otherwise(0)).alias('goods'),
        F.sum(F.when(F.col(target_col) == 1, 1).otherwise(0)).alias('bads')
    )
    
    # Calculate percentages and WoE
    woe_df = woe_df.withColumn('pct_goods', F.col('goods') / total_goods)
    woe_df = woe_df.withColumn('pct_bads', F.col('bads') / total_bads)
    
    # Add small constant to avoid log(0)
    woe_df = woe_df.withColumn('woe', 
        F.log((F.col('pct_goods') + 0.0001) / (F.col('pct_bads') + 0.0001)))
    
    # Calculate IV contribution
    woe_df = woe_df.withColumn('iv_contrib', 
        (F.col('pct_goods') - F.col('pct_bads')) * F.col('woe'))
    
    # Get total IV
    iv_total = woe_df.agg(F.sum('iv_contrib').alias('iv')).collect()[0]['iv']
    
    return woe_df, iv_total

# Calculate WoE for key categorical features
categorical_features = ['delinquency_category', 'age_bin', 'income_bin', 'interest_rate_bin']

woe_results = {}
iv_summary = []

for feature in categorical_features:
    woe_df, iv_value = calculate_woe_iv(df_features, feature)
    woe_results[feature] = woe_df
    iv_summary.append((feature, iv_value))

# Display IV summary
iv_summary_df = spark.createDataFrame(iv_summary, ['Feature', 'IV'])
iv_summary_df = iv_summary_df.withColumn('Strength',
    F.when(F.col('IV') >= 0.3, 'Strong')
     .when(F.col('IV') >= 0.1, 'Medium')
     .when(F.col('IV') >= 0.02, 'Weak')
     .otherwise('Useless')
).orderBy(F.desc('IV'))

display(iv_summary_df)

Feature,IV,Strength
delinquency_category,0.3943989654037333,Strong
interest_rate_bin,0.3622481002442093,Strong
income_bin,0.31038773088903765,Strong
age_bin,0.23194423764773237,Medium


In [0]:
# Feature Engineering: Binning and transformations
from pyspark.sql.types import DoubleType

# Create binned features for continuous variables
df_features = df_silver

# Age bins (21-30, 31-40, 41-50, 51+)
df_features = df_features.withColumn('age_bin',
    F.when(F.col('age') <= 30, '21-30')
     .when(F.col('age') <= 40, '31-40')
     .when(F.col('age') <= 50, '41-50')
     .otherwise('51+'))

# Annual income bins (quintiles based on distribution)
df_features = df_features.withColumn('income_bin',
    F.when(F.col('annual_income') < 35000, '<35K')
     .when(F.col('annual_income') < 50000, '35K-50K')
     .when(F.col('annual_income') < 70000, '50K-70K')
     .when(F.col('annual_income') < 100000, '70K-100K')
     .otherwise('100K+'))

# Interest rate bins (risk-based tiers)
df_features = df_features.withColumn('interest_rate_bin',
    F.when(F.col('interest_rate') < 8, 'Low_<8%')
     .when(F.col('interest_rate') < 12, 'Medium_8-12%')
     .when(F.col('interest_rate') < 16, 'High_12-16%')
     .otherwise('VeryHigh_16%+'))

# Behavioral flags
df_features = df_features.withColumn('high_utilization_flag', 
    (F.col('credit_utilisation_pct') > 75).cast('int'))

df_features = df_features.withColumn('new_account_flag',
    (F.col('months_since_oldest_account') < 60).cast('int'))

df_features = df_features.withColumn('frequent_inquiries_flag',
    (F.col('num_hard_inquiries_6mo') >= 3).cast('int'))

df_features = df_features.withColumn('high_dti_flag',
    (F.col('dti_ratio') > 0.4).cast('int'))

# Keep original numeric features for selected strong/medium variables
selected_numeric = [
    'interest_rate', 'annual_income', 'age',
    'num_delinquencies_2yr', 'months_since_oldest_account',
    'employment_length_years', 'months_since_last_delinquency',
    'dti_ratio', 'credit_utilisation_pct', 'num_open_accounts',
    'loan_amount', 'num_hard_inquiries_6mo', 'total_revolving_balance',
    'pct_accounts_current'
]

feature_summary = df_features.select('applicant_id_hash', 'default_flag', 'set', 
                                      'delinquency_category', 'age_bin', 'income_bin',
                                      'interest_rate_bin', 'high_utilization_flag',
                                      'new_account_flag', 'frequent_inquiries_flag',
                                      'high_dti_flag').limit(5)

display(feature_summary)

applicant_id_hash,default_flag,set,delinquency_category,age_bin,income_bin,interest_rate_bin,high_utilization_flag,new_account_flag,frequent_inquiries_flag,high_dti_flag
0fe333e94739d59b,0,train,Moderate_12-36mo,41-50,<35K,High_12-16%,0,0,0,0
8efd7b8ba3b54551,1,train,Old_36mo+,41-50,<35K,High_12-16%,0,0,0,0
b17623a910198f99,0,train,Moderate_12-36mo,21-30,50K-70K,High_12-16%,0,1,0,0
14d2376e8b32a216,1,train,Recent_0-12mo,31-40,<35K,High_12-16%,0,0,0,0
d2c6312855b2d028,0,train,Recent_0-12mo,41-50,<35K,High_12-16%,0,0,1,0


In [0]:
# Summary statistics for strong and medium features
strong_features = ['interest_rate', 'annual_income']
medium_features = ['age', 'num_delinquencies_2yr', 'months_since_oldest_account', 
                   'employment_length_years', 'months_since_last_delinquency']

# Get descriptive stats
features_to_analyze = strong_features + medium_features
stats_df = df_silver.select(features_to_analyze + ['default_flag']).summary('count', 'mean', 'stddev', 'min', 'max')
display(stats_df)

summary,interest_rate,annual_income,age,num_delinquencies_2yr,months_since_oldest_account,employment_length_years,months_since_last_delinquency,default_flag
count,120358,120358,120358,120358,120358,120358,120358,120358
mean,12.004823194137469,67197.92748300903,38.41983083799997,0.8226291563502218,177.4949982552053,5.811797304707623,11.463758121603881,0.1543644793034115
stddev,2.9512854726184905,93494.40587425814,11.136705299965255,1.2434278058151413,84.14891814798011,4.173130616588458,23.934253435693417,0.3612992823227223
min,5.0,2909.0,21,0,9.0,0.0,-1.0,0
max,27.72,2000000.0,70,9,537.0,30.4,180.0,1


In [0]:
# Load silver layer data and investigate patterns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

# Load silver layer data
df_silver = spark.table('dataanalytics.ml1.loan_book_silver')

# Basic statistics
total_count = df_silver.count()
train_count = df_silver.filter(F.col('set') == 'train').count()
test_count = df_silver.filter(F.col('set') == 'test').count()

default_rate_overall = df_silver.agg(F.mean('default_flag').alias('default_rate')).collect()[0]['default_rate']
default_rate_train = df_silver.filter(F.col('set') == 'train').agg(F.mean('default_flag').alias('default_rate')).collect()[0]['default_rate']
default_rate_test = df_silver.filter(F.col('set') == 'test').agg(F.mean('default_flag').alias('default_rate')).collect()[0]['default_rate']

# Create summary dataframe
summary_data = [
    ('Total Records', total_count),
    ('Training Set', train_count),
    ('Test Set', test_count),
    ('Overall Default Rate', f'{default_rate_overall:.2%}'),
    ('Train Default Rate', f'{default_rate_train:.2%}'),
    ('Test Default Rate', f'{default_rate_test:.2%}')
]

summary_df = spark.createDataFrame(summary_data, ['Metric', 'Value'])
display(summary_df)

Metric,Value
Total Records,120358
Training Set,84274
Test Set,36084
Overall Default Rate,15.44%
Train Default Rate,15.45%
Test Default Rate,15.41%


In [0]:
# Examine delinquency_category distribution (strong categorical feature)
delinquency_dist = df_silver.groupBy('delinquency_category').agg(
    F.count('*').alias('count'),
    F.mean('default_flag').alias('default_rate')
).orderBy(F.desc('count'))

display(delinquency_dist)

delinquency_category,count,default_rate
No_History,60068,0.0795098887927016
Recent_0-12mo,29646,0.2760912096066923
Moderate_12-36mo,16205,0.20617093489663685
Old_36mo+,14439,0.15769790151672552
